# 01 ASR pairs + synthesis  `[GPU — L4 is enough]`
Real ASR pairs first (the pipeline's bulk decode: 6 decodes per clip with N-best,
CUDA sherpa-onnx), then synthetic transcripts → voice-cloned TTS → synthetic pairs
→ leakage report (former stages 02, 04–07). Long poles: pair decode + viXTTS, both
resume per item.

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

def _token():
    # Colab Secrets live in userdata, NOT os.environ - check both.
    for key in ('CAREPATH_GITHUB_TOKEN', 'GITHUB_TOKEN'):
        if os.environ.get(key):
            return os.environ[key]
    try:
        from google.colab import userdata
        for key in ('CAREPATH_GITHUB_TOKEN', 'GITHUB_TOKEN'):
            try:
                val = userdata.get(key)
                if val:
                    return val
            except Exception:
                pass
    except Exception:
        pass
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    target = Path('/content/carepath')
    if _find(target):                       # already cloned in this runtime
        REPO = target
    else:
        if target.exists():
            shutil.rmtree(target)           # remove a half-cloned leftover
        url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
        tok = _token()
        if tok and url.startswith('https://github.com/'):
            url = url.replace('https://', f'https://x-access-token:{tok}@')
        r = subprocess.run(['git', 'clone', url, str(target)], capture_output=True, text=True)
        if r.returncode != 0:
            err = (r.stderr or r.stdout)
            if tok:
                err = err.replace(tok, '***')
            raise SystemExit(
                'git clone failed. This repo is private — add a Colab Secret named '
                'GITHUB_TOKEN (key icon in the left sidebar, toggle "Notebook access") '
                'holding a GitHub token with read access to the repo, then re-run.\n\n' + err)
        REPO = target
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = os.environ.get('CAREPATH_PROFILE', 'full')  # default full; set CAREPATH_PROFILE=smoke for a plumbing-only check
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'coqui-tts'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'sherpa-onnx==1.13.3+cuda12.cudnn9',
                '-f', 'https://k2-fsa.github.io/sherpa/onnx/cuda.html'])


# Stage 02: Real GEC pairs + N-best + error-signal report  `[GPU]`
Paper §3.1 — run Gipformer (or mock for smoke) over ViMedCSS audio to build
`raw_asr -> gold_text` pairs. `--n-best` adds the perturbation hypotheses (paper
§4.3), so the full run decodes each clip 6x — that is the pipeline's bulk decode
and it needs the CUDA provider (weeks on a 2-vCPU runtime, hours on an L4). The
**error-signal report** warns if the ASR is too accurate on train to teach the
corrector (paper §3.2).

In [ ]:
CTX.restore([str(P.datastore)])  # built in the CPU data-prep notebook
pairs_out = CTX.durable(P.real_pairs)  # write straight to Drive on Colab so --resume survives a disconnect
CTX.run_step(['scripts/gec/make_pairs.py', '--dataset', CTX.dataset, '--output', pairs_out,
              '--asr-provider', PROF.asr_provider, '--datastore', str(P.datastore),
              '--retrieval-backend', PROF.retrieval_backend,
              '--limit-per-split', str(PROF.limit_per_split or 0),
              '--n-best', str(PROF.n_best), '--resume'],
             env_extra={'GIPFORMER_PROVIDER': os.environ.get('GIPFORMER_PROVIDER', 'cuda')})
from carepath.gec.data import read_jsonl
from carepath.gec.evaluate import train_error_signal
print(train_error_signal(read_jsonl(pairs_out)))


# Stage 04: Synthetic in-domain transcripts  `[GPU-light]`
Paper §4.1 Step 1 — few-shot an open LLM for new in-domain transcripts, with the
n-gram leakage guard rejecting near-copies. `synth_count=None` (full) matches the
real train size (nsyn = n).

In [ ]:
# Pull inputs from Drive: no-op in the same session, lets a fresh runtime
# (or a teammate's Colab) resume mid-notebook.
CTX.restore([str(P.datastore), str(P.real_pairs)])
from carepath.gec.data import read_jsonl
count = PROF.synth_count or (sum(1 for r in read_jsonl(P.real_pairs) if r.get('split') == 'train') or 50)
args = ['scripts/gec/gen_synthetic.py', '--pairs', str(P.real_pairs),
        '--output', CTX.durable(P.synth_clean), '--count', str(count)]  # durable: persist so a later disconnect won't regenerate
if PROF.name != 'smoke':
    args.append('--load-in-4bit')
CTX.run_step(args)


# Stage 05: Voice-cloning TTS  `[GPU]`
Paper §4.1 Step 2 / App. D — synthesize speech with viXTTS conditioned on random
in-domain reference clips (falls back to single-speaker MMS, labeled as such).

In [ ]:
args = ['scripts/gec/voice_clone_tts.py', '--input', CTX.durable(P.synth_clean),
        '--output', CTX.durable(P.tts_manifest), '--provider', PROF.tts_provider,  # durable: viXTTS is long, survive a disconnect
        '--ref-dataset', CTX.dataset, '--ref-count', '20', '--resume']
if PROF.synth_tts_limit:
    args += ['--limit', str(PROF.synth_tts_limit)]
CTX.run_step(args)


# Stage 06: Synthetic GEC pairs (+ N-best)  `[CPU/GPU]`
Paper §4.1 Step 3 — run the ASR over the voice-cloned audio to get synthetic
`raw_asr -> gold_text` pairs, with the same perturbation N-best as the real pairs.

In [ ]:
CTX.run_step(['scripts/gec/make_synth_pairs.py', '--input', CTX.durable(P.tts_manifest),
              '--output', CTX.durable(P.synth_pairs), '--datastore', str(P.datastore),
              '--n-best', str(PROF.n_best), '--resume'],  # durable: survive a disconnect, no re-ASR
             env_extra={'GIPFORMER_PROVIDER': os.environ.get('GIPFORMER_PROVIDER', 'cuda')})


# Stage 07: Leakage report — in-domain but not memorized  `[GPU-light]`
Paper App. C / Table 6 — SentenceBERT cosine + BLEU of synthetic vs real. High
cosine + low BLEU means the synthetic data is on-domain without copying.

In [ ]:
CTX.run_step(['scripts/gec/check_leakage.py', '--synthetic', CTX.durable(P.synth_clean),
              '--real', str(P.real_pairs), '--output', str(P.leakage)])
import json
print(json.load(open(P.leakage, encoding='utf-8')))
